[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Implement **Batch Normalization** with both **training** and **inference** behavior.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed **across the batch** (dim=0).

In inference mode, use the provided **running mean/var** instead of current batch stats.

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm1d`, etc.
- **Normalize** with the batch mean and the **biased** batch variance over `dim=0` (`unbiased=False`)
- **Update** the running stats like `nn.BatchNorm1d`, using the **unbiased** batch variance for `running_var`:
  - `running_mean = (1 - momentum) * running_mean + momentum * batch_mean`
  - `running_var = (1 - momentum) * running_var + momentum * x.var(dim=0, unbiased=True)`
  - the batch has `N >= 2` rows (with `N = 1` the unbiased variance is undefined, and PyTorch raises an error)
- Use `running_mean` / `running_var` for inference when `training=False`
- Must support autograd w.r.t. `x`, `gamma`, `beta` (running statistics should be treated as buffers, not parameters requiring gradients)

### Why two different variances?
They estimate different things:
- **Normalizing** uses the spread of *this exact batch*, so dividing by $N$ is exactly right.
- The **running variance** is meant to estimate the variance of the *whole data distribution* for use at inference. A batch's own spread systematically underestimates that, and dividing by $N - 1$ (Bessel's correction) removes the bias. With batches of 8 the difference is $8/7 \approx 14\%$.

> **Conventions differ:** PyTorch applies the correction to the running variance. Flax and Keras use the biased batch variance for both. This problem follows PyTorch, so your output should match `nn.BatchNorm1d` exactly.

### 📐 Notation

| Symbol | Meaning |
|:---:|---|
| `N` | batch size (number of rows) |
| `D` | number of features |


📖 **Reference solution:** [`07_batchnorm_solution.ipynb`](07_batchnorm_solution.ipynb) (JupyterLab) · [`solutions/07_batchnorm_solution.ipynb`](../solutions/07_batchnorm_solution.ipynb) (GitHub / VS Code). Try it yourself first!

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    pass  # Replace this

In [ ]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")